In [ ]:
#!pip install -q numpy
#!pip install -q transformers datasets evaluate accelerate wandb

In [ ]:
# If in Colab, then install required libraries
if 'google.colab' in str(get_ipython()):
    !pip install numpy -U -qq
    !pip install transformers datasets accelerate evaluate wandb trl peft bitsandbytes -U -qq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 38.7 MB/s eta 0:00:00


In [ ]:
# standard data science librraies for data handling and v isualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
import gc
import time

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import multilabel_confusion_matrix, precision_score, recall_score, f1_score
import joblib

import torch
import torch.nn as nn
import ast


# New libraries introduced in this notebook
import evaluate
from datasets import load_dataset, DatasetDict, Dataset, ClassLabel
from transformers import (
    TrainingArguments,
    Trainer,
    set_seed,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    pipeline,
    BitsAndBytesConfig,
)

from trl import SFTTrainer
from peft import (
    TaskType,
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
    AutoPeftModelForSequenceClassification,
    PeftConfig
)
import wandb
from google.colab import userdata
from huggingface_hub import login, HfApi, create_repo

In [ ]:
set_seed(42)

In [ ]:
wandb_api_key = userdata.get('WANDB_API_KEY')
hf_token = userdata.get('HF_TOKEN')

In [ ]:
if hf_token:
    # Log in to Hugging Face
    login(token=hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("Hugging Face token not found in notebook secrets.")


Successfully logged in to Hugging Face!


In [ ]:
if wandb_api_key:
  wandb.login(key=wandb_api_key)
  print("Successfully logged in to WANDB!")
else:
    print("WANDB key not found in notebook secrets.")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: stevenqcly (stevenqcly-the-university-of-texas-at-dallas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged in to WANDB!


In [ ]:
def free_gpu_memory():
    """
    Frees up GPU memory after CUDA out-of-memory error in Colab.

    This function performs the following steps:
    1. Deletes all PyTorch objects to clear references.
    2. Calls garbage collection to remove unreferenced objects from memory.
    3. Uses torch.cuda.empty_cache() to release cached GPU memory.
    4. Waits for a moment to ensure memory is fully released.
    """
    try:
        # Delete all torch tensors to free up memory
        for obj in list(locals().values()):
            if torch.is_tensor(obj):
                del obj

        # Collect garbage to release any remaining unused memory
        gc.collect()

        # Empty the CUDA cache to release GPU memory
        torch.cuda.empty_cache()

        # Adding a small delay to allow memory to be fully released
        time.sleep(2)

        print("GPU memory has been freed.")
    except Exception as e:
        print(f"Error while freeing GPU memory: {e}")

In [ ]:
free_gpu_memory()

GPU memory has been freed.


In [ ]:
from pathlib import Path
# Determine the storage location based on the execution environment
# If running on Google Colab, use Google Drive as storage
if 'google.colab' in str(get_ipython()):
    from google.colab import drive  # Import Google Drive mounting utility
    drive.mount('/content/drive')  # Mount Google Drive

    # Set base folder path for storing data on Google Drive
    base_folder= Path('/content/drive/My Drive/Fall_2025_Notes/NLP/HW 6')
    project_folder = Path('/content/drive/My Drive/Fall_2025_Notes/NLP/HW 6')
# If running locally, specify a different path
else:
    # Set base folder path for storing data on local machine
    base_folder= Path('/home/')
    project_folder = Path('/home/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Students can set the data_folder and model_folder paths based on their base_folder setup.
data_folder = base_folder/'datasets'
model_folder = project_folder/'models'
model_folder.mkdir(exist_ok=True, parents = True)
data_folder.mkdir(exist_ok=True, parents = True)

In [ ]:
#Now import the train set
#Import the dataset
train_df = pd.read_csv(data_folder/"train.csv")
print(train_df.shape)
train_df.head()

(7724, 13)


,ID,Tweet,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust
0,2017-21441,“Worry is a down payment on a problem you may ...,0,1,0,0,0,0,1,0,0,0,1
1,2017-31535,Whatever you decide to do make sure it makes y...,0,0,0,0,1,1,1,0,0,0,0
2,2017-21068,@Max_Kellerman it also helps that the majorit...,1,0,1,0,1,0,1,0,0,0,0
3,2017-31436,Accept the challenges so that you can literall...,0,0,0,0,1,0,1,0,0,0,0
4,2017-22195,My roommate: it's okay that we can't spell bec...,1,0,1,0,0,0,0,0,0,0,0


In [ ]:
id_col = "ID"
text_col = "Tweet"

label_cols = [
    'anger',
    'anticipation',
    'disgust',
    'fear',
    'joy',
    'love',
    'optimism',
    'pessimism',
    'sadness',
    'surprise',
    'trust'
]

class_names = label_cols
num_labels = len(class_names)
print("Labels:", class_names)
print("num_labels:", num_labels)


Labels: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']
num_labels: 11


In [ ]:
#Transforming data to HF compatible dataset
train_df_hf = pd.DataFrame({
    "text": train_df[text_col],
    "labels": train_df[label_cols].values.tolist()
})

train_df_hf.head()


,text,labels
0,“Worry is a down payment on a problem you may ...,"[0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1]"
1,Whatever you decide to do make sure it makes y...,"[0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0]"
2,@Max_Kellerman it also helps that the majorit...,"[1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0]"
3,Accept the challenges so that you can literall...,"[0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0]"
4,My roommate: it's okay that we can't spell bec...,"[1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]"


In [ ]:
#Test/Train/Validation Split
hf_dataset = Dataset.from_pandas(train_df_hf)

splits = hf_dataset.train_test_split(test_size=0.4, seed=42)
train_split = splits["train"]
tmp_split   = splits["test"].train_test_split(test_size=0.5, seed=42)

train_dataset = train_split           # 60%
valid_dataset = tmp_split["train"]    # 20%
test_dataset  = tmp_split["test"]     # 20%

train_val_subset = {
    "train": train_dataset,
    "valid": valid_dataset,
    "test":  test_dataset,
}


In [ ]:
checkpoint = "google/gemma-2-2b"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

max_length = 128

def tokenize_fn(batch):
    return tokenizer(
        text=batch["text"],
        truncation=True,
        max_length=max_length,
    )

In [ ]:
#Apply tokenizer
tokenized_dataset = {
    split: ds.map(tokenize_fn, batched=True)
    for split, ds in train_val_subset.items()
}

for split in tokenized_dataset:
    tokenized_dataset[split] = tokenized_dataset[split].remove_columns(["text"])
    tokenized_dataset[split].set_format(type="torch")


Map:   0%|          | 0/4634 [00:00<?, ? examples/s]

Map:   0%|          | 0/1545 [00:00<?, ? examples/s]

Map:   0%|          | 0/1545 [00:00<?, ? examples/s]

In [ ]:
#Convert to float
def to_float_labels(example):
    labels = torch.tensor(example["labels"], dtype=torch.float)
    example["labels"] = labels
    return example

for split in tokenized_dataset:
    tokenized_dataset[split] = tokenized_dataset[split].map(to_float_labels)


Map:   0%|          | 0/4634 [00:00<?, ? examples/s]

/tmp/ipython-input-1663749540.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(example["labels"], dtype=torch.float)


Map:   0%|          | 0/1545 [00:00<?, ? examples/s]

Map:   0%|          | 0/1545 [00:00<?, ? examples/s]

In [ ]:
#Checking to make sure everything looks right
tokenized_dataset["train"][0]

{'labels': tensor([1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0]),
 'input_ids': tensor([     2, 235348, 176589,    477,   1469, 176589,    477,   1118,  50906,
          13626,   1547,    730, 235254,   1718, 235303, 235256,   2593,   2245,
            577,  23790,    777,   2430,    573,   9057,    608,  21302,   5294,
            675,   8909,    646, 235303,  22812,  90830]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1])}

In [ ]:
#QLoRA Setup
def get_appropriate_dtype():
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0) >= (8, 0):
        return torch.bfloat16
    return torch.float16

torch_dtype = get_appropriate_dtype()
torch_dtype

torch.bfloat16

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_quant_storage=torch_dtype,
)

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    quantization_config=bnb_config,
    torch_dtype=torch_dtype,
    trust_remote_code=True,
)

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = False

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/481M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of Gemma2ForSequenceClassification were not initialized from the model checkpoint at google/gemma-2-2b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
#LoRA COnfig
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=128,
    lora_alpha=256,
    lora_dropout=0.05,
    modules_to_save=["score"],
    target_modules=['v_proj', 'q_proj', 'up_proj', 'o_proj', 'down_proj', 'gate_proj', 'k_proj'],
)

lora_model = get_peft_model(base_model, peft_config)
lora_model.print_trainable_parameters()


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 166,159,104 || all params: 2,780,526,336 || trainable%: 5.9758


In [ ]:
#Weight function
def calculate_pos_weights(dataset_dict):
    # use the train split
    train_ds = dataset_dict["train"]
    first = train_ds[0]["labels"]
    num_labels = len(first)

    total_pos = np.zeros(num_labels)
    total_neg = np.zeros(num_labels)

    for ex in train_ds:
        labels = np.array(ex["labels"])
        total_pos += labels
        total_neg += 1 - labels

    pos_weights = total_neg / np.clip(total_pos, 1, None)
    return torch.tensor(pos_weights, dtype=torch.float32)

pos_weights = calculate_pos_weights(tokenized_dataset)
pos_weights


/tmp/ipython-input-3218730664.py:12: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  labels = np.array(ex["labels"])


tensor([ 1.6270,  5.9789,  1.6166,  4.6306,  1.7404,  8.7149,  2.4301,  7.6294,
         2.3006, 19.0606, 19.1478])

In [ ]:
#Metrics
f1_metric = evaluate.load("f1", "multilabel")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs > 0.5).astype(int)

    f1_macro = f1_metric.compute(
        predictions=preds,
        references=labels,
        average="macro"
    )["f1"]

    return {"f1_macro": f1_macro}

In [ ]:
#BCE Logits Trainer
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.get("logits")

        device = next(model.parameters()).device
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss


In [ ]:
#Training Arguments with WandB
model_folder = Path("/content/gemma_emotion_lora") if 'google.colab' in str(get_ipython()) else Path("./gemma_emotion_lora")
model_folder.mkdir(exist_ok=True, parents=True)

run_name = "emotion_multilabel_gemma_base"

use_fp16 = torch_dtype == torch.float16
use_bf16 = torch_dtype == torch.bfloat16


In [ ]:
# #OLd Train_args (can be improved!!!)
# training_args = TrainingArguments(
#     seed=42,
#     num_train_epochs=2,                     # can increase later
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     gradient_accumulation_steps=4,
#     gradient_checkpointing=True,
#     gradient_checkpointing_kwargs={"use_reentrant": False},
#     weight_decay=0.0,
#     learning_rate=1e-5,
#     optim="adamw_torch",

#     output_dir=str(model_folder),
#     eval_strategy="steps",
#     eval_steps=50,
#     save_strategy="steps",
#     save_steps=50,
#     save_total_limit=2,
#     load_best_model_at_end=True,
#     metric_for_best_model="f1_macro",
#     greater_is_better=True,

#     logging_strategy="steps",
#     logging_steps=20,
#     report_to="wandb",
#     run_name=run_name,

#     fp16=use_fp16,
#     bf16=use_bf16,
# )


In [ ]:
#New Training Args
training_args = TrainingArguments(
    seed=42,
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    gradient_checkpointing=False,

    learning_rate=2e-5,
    weight_decay=0.03,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    output_dir=str(model_folder),
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=20,
    report_to="wandb",
    run_name="PartA_decoder_cls_regularized",

    fp16=use_fp16,
    bf16=use_bf16,
)


In [ ]:
#New trainer with early stopping
trainer = CustomTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["valid"],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)


/tmp/ipython-input-2514084393.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  trainer = CustomTrainer(


In [ ]:
#Train + Evaluate
train_result = trainer.train()
eval_results = trainer.evaluate(tokenized_dataset["valid"])
eval_results

Step,Training Loss,Validation Loss,F1 Macro
100,1.346000,0.724577,0.556798
200,0.853500,0.818950,0.581974


{'eval_loss': 0.8189501166343689,
 'eval_f1_macro': 0.5819737740026827,
 'eval_runtime': 8.1267,
 'eval_samples_per_second': 190.113,
 'eval_steps_per_second': 6.029,
 'epoch': 4.0}

In [ ]:
best_step = None
best_f1 = -1.0

for log in trainer.state.log_history:
    if "eval_f1_macro" in log:
        step = log.get("step", None)
        f1 = log["eval_f1_macro"]
        if f1 > best_f1:
            best_f1 = f1
            best_step = step

print("Best eval_f1_macro:", best_f1)
print("Best step:", best_step)


Best eval_f1_macro: 0.5819737740026827
Best step: 200


In [ ]:
%env WANDB_PROJECT=emotion_decoder_hw6

wandb.init(project="emotion_decoder_hw6", name=run_name)

env: WANDB_PROJECT=emotion_decoder_hw6


eval/f1_macro,▁██
eval/loss,▁██
eval/runtime,▄▁█
eval/samples_per_second,▅█▁
eval/steps_per_second,▅█▁
train/epoch,▁▂▂▃▃▃▄▄▅▅▆▆▆▇▇███
train/global_step,▁▂▂▃▃▃▄▄▅▅▆▆▆▇▇███
train/grad_norm,█▄▄▂▂▂▂▃▂▂▁▂▁▁
train/learning_rate,▅██▇▇▆▅▅▄▃▂▂▁▁
train/loss,█▄▃▃▂▂▂▂▁▁▁▁▁▁
eval/f1_macro,0.58197


In [ ]:
from sklearn.metrics import classification_report
#Training report
logits, labels = trainer.predict(tokenized_dataset["valid"])[:2]
probs = torch.sigmoid(torch.tensor(logits)).numpy()
preds = (probs > 0.5).astype(int)

print("Validation F1 macro:",
      f1_score(labels, preds, average="macro"))
print(classification_report(labels, preds, target_names=class_names))

Validation F1 macro: 0.5819737740026827
              precision    recall  f1-score   support

       anger       0.74      0.85      0.79       534
anticipation       0.28      0.56      0.38       220
     disgust       0.74      0.78      0.76       573
        fear       0.61      0.83      0.70       275
         joy       0.82      0.83      0.83       597
        love       0.45      0.69      0.54       180
    optimism       0.63      0.79      0.70       454
   pessimism       0.33      0.57      0.42       189
     sadness       0.61      0.77      0.68       442
    surprise       0.27      0.66      0.38        70
       trust       0.14      0.54      0.23        70

   micro avg       0.58      0.77      0.66      3604
   macro avg       0.51      0.72      0.58      3604
weighted avg       0.63      0.77      0.68      3604
 samples avg       0.60      0.77      0.64      3604



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#Saving model and tokenizer
trainer.save_model(model_folder)
tokenizer.save_pretrained(model_folder)

('/content/gemma_emotion_lora/tokenizer_config.json',
 '/content/gemma_emotion_lora/special_tokens_map.json',
 '/content/gemma_emotion_lora/tokenizer.model',
 '/content/gemma_emotion_lora/added_tokens.json',
 '/content/gemma_emotion_lora/tokenizer.json')

In [ ]:
# After training
inference_model = trainer.model  # <- reuse the trained PEFT model
inference_model.eval()

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): Gemma2ForSequenceClassification(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=128, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=128, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleD

In [ ]:
# inference_model = AutoPeftModelForSequenceClassification.from_pretrained(
#     model_folder,
#     quantization_config=bnb_config,
#     torch_dtype=torch_dtype,
# )
# inference_model.eval()

## Inference

In [ ]:
#Import test set now
test_df = pd.read_csv(data_folder/"test.csv")
test_df.head()

,ID,Tweet,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust
0,2018-01559,@Adnan__786__ @AsYouNotWish Dont worry Indian ...,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE
1,2018-03739,"Academy of Sciences, eschews the normally sobe...",NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE
2,2018-00385,I blew that opportunity -__- #mad,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE
3,2018-03001,This time in 2 weeks I will be 30... 😥,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE
4,2018-01988,#Deppression is real. Partners w/ #depressed p...,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE,NONE


In [ ]:
test_texts = test_df[text_col].tolist()
test_ids   = test_df[id_col].tolist()

test_hf = Dataset.from_dict({"text": test_texts})
test_tokenized = test_hf.map(tokenize_fn, batched=True)
test_tokenized = test_tokenized.remove_columns(["text"])
test_tokenized.set_format(type="torch")

Map:   0%|          | 0/3259 [00:00<?, ? examples/s]

In [ ]:
def predict_dataset(model, dataset, threshold=0.5):
    pred_trainer = Trainer(model=model, tokenizer=tokenizer)
    outputs = pred_trainer.predict(dataset)
    logits = outputs.predictions
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs > threshold).astype(int)
    return preds, probs

test_preds, test_probs = predict_dataset(inference_model, test_tokenized)

/tmp/ipython-input-2260328784.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  pred_trainer = Trainer(model=model, tokenizer=tokenizer)


In [ ]:
submission = pd.DataFrame(test_preds, columns=label_cols)
submission.insert(0, id_col, test_ids)
submission.head()

,ID,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust
0,2018-01559,1,0,0,1,0,0,1,0,0,0,0
1,2018-03739,0,0,1,0,0,0,0,0,0,0,0
2,2018-00385,1,0,1,0,0,0,0,0,1,0,0
3,2018-03001,0,0,0,1,0,0,0,1,1,0,0
4,2018-01988,0,0,0,1,0,0,0,1,1,0,0


In [ ]:
submission.to_csv("submission_partA.csv", index=False)

In [ ]:
from google.colab import files
files.download("submission_partA.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>